# Statistical Analysis of Benchmark Results (`run_benchmark`)

This notebook provides a comprehensive statistical analysis of evaluation results produced by `run_benchmark.py`:
- **Three-Way Ablation**: Quantitative comparison across **Baseline** (single-pass), **TTA Pure** (averaged heatmaps with native decoding), and **Proposed** (TTA + Robust GMM + MRF).
- **Topological Error Quantification**: Analysis of limb swaps under loose (`margin=0.0`) and strict (`margin=0.7`) anatomical thresholds.
- **Diagnostic Focus Groups**: Verification of cohort distributions (`Wins`, `Regressions`, `High_Uncertainty`, `Edge_Cases`) and consistency with `focus_groups_ids.json`.
- **Per-Keypoint Outlier Diagnostics**: Analysis of `Top_Single_KP_Gains` and `Worst_Single_KP_Drops` from `keypoint_outliers_ids.json`.
- **Bimodal Uncertainty Dynamics**: Distribution of GMM mixture weights ($\pi_1, \pi_2$) across ambiguous anatomical joints.


In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root dynamically
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd().parent
)
PROJECT_ROOT = project_root
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
mmpose_root = project_root / "models" / "mmpose"
if mmpose_root.exists() and str(mmpose_root) not in sys.path:
    sys.path.insert(0, str(mmpose_root))

print(f"Project root: {project_root}")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10

In [ ]:
# Locate benchmark runs containing results_metadata.parquet
outputs_root = project_root / 'outputs'
runs_with_parquet = []

for day in sorted(outputs_root.glob('*')):
    if not day.is_dir():
        continue
    for run in sorted(day.glob('*')):
        pq = run / 'results_metadata.parquet'
        if run.is_dir() and pq.exists():
            runs_with_parquet.append(run)

print(f"Total benchmark runs with parquet found: {len(runs_with_parquet)}")
for r in runs_with_parquet[-8:]:
    print(f"  - {r.relative_to(project_root)}")

# Automatically select the latest run with parquet, with fallback
if runs_with_parquet:
    run_dir = runs_with_parquet[-1]
else:
    run_dir = project_root / 'outputs' / '2026-02-23' / '11-22-15'

print(f"\nActive Run Selected: {run_dir.relative_to(project_root) if run_dir.exists() else run_dir}")


In [ ]:
parquet_path = run_dir / 'results_metadata.parquet'
assert parquet_path.exists(), f"Parquet file not found: {parquet_path}"

df = pd.read_parquet(parquet_path)
print(f"Dataset: {df['dataset'].iloc[0] if 'dataset' in df.columns else 'N/A'}")
print(f"Total Evaluated Instances: {len(df):,}")
print(f"Available Metric Columns: {len(df.columns)}")
display(df.head(3))


In [ ]:
# -------------------------------------------------------------------------
# 1. Three-Way Ablation Study: Baseline vs TTA vs TTA+GMM/MRF
# -------------------------------------------------------------------------
eval_cols = ['oks_base', 'oks_tta', 'oks_ours', 'delta_oks', 'delta_oks_tta', 'delta_oks_ours_over_tta']
present_cols = [c for c in eval_cols if c in df.columns]

ablation_summary = []
for c in present_cols:
    ablation_summary.append({
        'Metric': c,
        'Mean': df[c].mean(),
        'Std': df[c].std(),
        'Median': df[c].median(),
        'IQR': df[c].quantile(0.75) - df[c].quantile(0.25),
        'Min': df[c].min(),
        'Max': df[c].max()
    })

ablation_df = pd.DataFrame(ablation_summary).set_index('Metric')
print("=== Three-Way Ablation Metrics Summary ===")
display(ablation_df.round(4))

if 'oks_base' in df.columns and 'oks_ours' in df.columns:
    oks_base_mean = df['oks_base'].mean()
    delta_mean = df['delta_oks'].mean()
    prop_gain = (delta_mean / oks_base_mean * 100) if oks_base_mean > 0 else 0.0
    print(f"Relative Accuracy Gain (Ours over Baseline): {prop_gain:+.2f}%")


In [ ]:
# -------------------------------------------------------------------------
# 2. Topological Error Quantification: Limb Swaps (Standard vs Strict)
# -------------------------------------------------------------------------
swap_metrics = {}

# Standard limb swaps (margin = 0.0)
std_swap_cols = ['swaps_base', 'swaps_tta', 'swaps_ours', 'swaps_corrected', 'swaps_introduced', 'swaps_corrected_tta', 'swaps_introduced_tta']
for c in std_swap_cols:
    if c in df.columns:
        swap_metrics[c] = int(df[c].sum())

# Strict limb swaps (margin = 0.7)
strict_swap_cols = ['swaps_base_strict', 'swaps_tta_strict', 'swaps_ours_strict', 'swaps_corrected_strict', 'swaps_introduced_strict']
for c in strict_swap_cols:
    if c in df.columns:
        swap_metrics[c] = int(df[c].sum())

if swap_metrics:
    swap_df = pd.DataFrame(list(swap_metrics.items()), columns=['Limb Swap Metric', 'Total Occurrences (Dataset Sum)'])
    print("=== Topological Limb Swap Analysis ===")
    display(swap_df)
    
    if 'swaps_base' in swap_metrics and 'swaps_ours' in swap_metrics:
        net_reduct = (swap_metrics['swaps_base'] - swap_metrics['swaps_ours']) / max(1, swap_metrics['swaps_base']) * 100
        print(f"Standard Limb Swap Net Reduction: {net_reduct:+.1f}%")
    if 'swaps_base_strict' in swap_metrics and 'swaps_ours_strict' in swap_metrics:
        net_reduct_strict = (swap_metrics['swaps_base_strict'] - swap_metrics['swaps_ours_strict']) / max(1, swap_metrics['swaps_base_strict']) * 100
        print(f"Strict (margin=0.7) Limb Swap Net Reduction: {net_reduct_strict:+.1f}%")
else:
    print("Limb swap columns not present in this run metadata.")


In [ ]:
# -------------------------------------------------------------------------
# 3. Diagnostic Focus Groups & Per-Keypoint Outliers
# -------------------------------------------------------------------------
OKS_IMPROVEMENT_THRESHOLD = 0.1
OKS_FAILURE_THRESHOLD = 0.5
COV_VOL_QUANTILE = 0.90

wins_mask = df['oks_ours'] > (df['oks_base'] + OKS_IMPROVEMENT_THRESHOLD)
reg_mask = df['oks_base'] > (df['oks_ours'] + OKS_IMPROVEMENT_THRESHOLD)
cov_q90 = df['covariance_vol'].quantile(COV_VOL_QUANTILE) if 'covariance_vol' in df.columns else np.nan
hu_mask = (df['covariance_vol'] > cov_q90) if 'covariance_vol' in df.columns else pd.Series(False, index=df.index)
ec_mask = (df['oks_base'] < OKS_FAILURE_THRESHOLD) & (df['oks_ours'] < OKS_FAILURE_THRESHOLD)

cohort_df = pd.DataFrame({
    'Cohort': ['Wins (ΔOKS > +0.1)', 'Regressions (ΔOKS < -0.1)', 'High Uncertainty (CovVol > Q90)', 'Edge Cases (Base & Ours < 0.5)'],
    'Total Eligible': [int(wins_mask.sum()), int(reg_mask.sum()), int(hu_mask.sum()), int(ec_mask.sum())],
    'Percentage (%)': [
        100 * wins_mask.sum() / len(df),
        100 * reg_mask.sum() / len(df),
        100 * hu_mask.sum() / len(df),
        100 * ec_mask.sum() / len(df),
    ]
})

# Cross-reference with focus_groups_ids.json
focus_json_path = run_dir / 'focus_groups_ids.json'
if focus_json_path.exists():
    with open(focus_json_path, 'r', encoding='utf-8') as f:
        focus_data = json.load(f)
    cohort_df['Saved for Deep Profiling'] = [
        len(focus_data.get('Wins', [])),
        len(focus_data.get('Regressions', [])),
        len(focus_data.get('High_Uncertainty', [])),
        len(focus_data.get('Edge_Cases', [])),
    ]
    print(f"Loaded focus group IDs from {focus_json_path.name}")

display(cohort_df.round(2))

# Keypoint Outliers (Top Gains / Worst Drops)
kp_outliers_path = run_dir / 'keypoint_outliers_ids.json'
if kp_outliers_path.exists():
    with open(kp_outliers_path, 'r', encoding='utf-8') as f:
        kp_outliers = json.load(f)
    print(f"\nPer-Keypoint Outliers Registered:")
    print(f"  - Top Single KP Gains: {len(kp_outliers.get('Top_Single_KP_Gains', []))} samples")
    print(f"  - Worst Single KP Drops: {len(kp_outliers.get('Worst_Single_KP_Drops', []))} samples")


In [ ]:
# -------------------------------------------------------------------------
# 4. Publication-Quality Diagnostic Plots (Q1 Standard)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Panel A: Accuracy Distribution Comparison (KDE)
ax_a = axes[0, 0]
if 'oks_base' in df.columns:
    sns.kdeplot(df['oks_base'], ax=ax_a, label='Baseline (Single-Pass)', color='#d62728', lw=2)
if 'oks_tta' in df.columns:
    sns.kdeplot(df['oks_tta'], ax=ax_a, label='TTA Pure (Ablation)', color='#ff7f0e', lw=2, linestyle=':')
if 'oks_ours' in df.columns:
    sns.kdeplot(df['oks_ours'], ax=ax_a, label='Proposed (TTA+GMM+MRF)', color='#1f77b4', lw=2.5)
ax_a.set_title('(a) OKS Accuracy Distribution', fontweight='bold')
ax_a.set_xlabel('Object Keypoint Similarity (OKS)')
ax_a.set_ylabel('Density')
ax_a.legend(frameon=True)
ax_a.set_xlim(0, 1)

# Panel B: Parity Scatter Plot (Baseline vs Proposed)
ax_b = axes[0, 1]
if {'oks_base', 'oks_ours'}.issubset(df.columns):
    sns.scatterplot(data=df, x='oks_base', y='oks_ours', ax=ax_b, s=16, alpha=0.35, color='#1f77b4', edgecolor='none')
    ax_b.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Parity (y = x)')
    ax_b.fill_between([0, 1], [0.1, 1.1], [0, 1], color='green', alpha=0.08, label='Win Region (Δ > +0.1)')
    ax_b.fill_between([0, 1], [0, 1], [-0.1, 0.9], color='red', alpha=0.08, label='Regression (Δ < -0.1)')
    ax_b.set_title('(b) Baseline vs Proposed Accuracy', fontweight='bold')
    ax_b.set_xlabel('Baseline OKS')
    ax_b.set_ylabel('Proposed OKS')
    ax_b.set_xlim(0, 1)
    ax_b.set_ylim(0, 1)
    ax_b.legend(loc='lower right', frameon=True, fontsize=9)

# Panel C: Uncertainty vs Error Residuals
ax_c = axes[1, 0]
if {'entropy', 'delta_oks'}.issubset(df.columns):
    sns.scatterplot(data=df, x='entropy', y='delta_oks', ax=ax_c, s=16, alpha=0.35, color='#2ca02c', edgecolor='none')
    ax_c.axhline(0.0, color='black', linestyle='--', lw=1)
    ax_c.set_title('(c) Predictive Heatmap Entropy vs ΔOKS Improvement', fontweight='bold')
    ax_c.set_xlabel('Predictive Entropy (nats)')
    ax_c.set_ylabel('ΔOKS (Ours - Baseline)')

# Panel D: Limb Swap Comparison
ax_d = axes[1, 1]
if 'swaps_base' in swap_metrics and 'swaps_ours' in swap_metrics:
    categories = ['Standard\n(margin=0.0)']
    base_vals = [swap_metrics.get('swaps_base', 0)]
    tta_vals = [swap_metrics.get('swaps_tta', 0)]
    ours_vals = [swap_metrics.get('swaps_ours', 0)]
    
    if 'swaps_base_strict' in swap_metrics:
        categories.append('Strict\n(margin=0.7)')
        base_vals.append(swap_metrics.get('swaps_base_strict', 0))
        tta_vals.append(swap_metrics.get('swaps_tta_strict', 0))
        ours_vals.append(swap_metrics.get('swaps_ours_strict', 0))
        
    x = np.arange(len(categories))
    w = 0.25
    ax_d.bar(x - w, base_vals, width=w, label='Baseline', color='#d62728', alpha=0.85)
    ax_d.bar(x, tta_vals, width=w, label='TTA Pure', color='#ff7f0e', alpha=0.85)
    ax_d.bar(x + w, ours_vals, width=w, label='Proposed (MRF)', color='#1f77b4', alpha=0.85)
    ax_d.set_xticks(x)
    ax_d.set_xticklabels(categories)
    ax_d.set_title('(d) Total Limb Swaps by Method', fontweight='bold')
    ax_d.set_ylabel('Total Swap Events')
    ax_d.legend(frameon=True)
else:
    ax_d.text(0.5, 0.5, 'Limb swap data not recorded in run', ha='center', va='center')

plt.tight_layout()
plt.show()


In [ ]:
# -------------------------------------------------------------------------
# 5. Salient Case Extraction for Qualitative Profiling
# -------------------------------------------------------------------------
inspect_cols = [c for c in ['image_id', 'oks_base', 'oks_tta', 'oks_ours', 'delta_oks', 'swaps_base', 'swaps_ours', 'entropy', 'covariance_vol'] if c in df.columns]

print("Top 5 Landmark Wins (Highest ΔOKS Gains):")
display(df.sort_values('delta_oks', ascending=False)[inspect_cols].head(5).round(4))

print("\nTop 5 Regressions (Failure Mode Analysis):")
display(df.sort_values('delta_oks', ascending=True)[inspect_cols].head(5).round(4))
